# Test Scores Analysis
This notebook analyzes test scores from `test_scores.csv`. The metrics in the CSV are stored as string tuples formatted like `(score, weight)`, so this notebook parses them to calculate accurate weighted averages.

In [2]:
import pandas as pd
import ast

# Path to the data (relative to the eval/ folder)
file_path = "../exp_output/nova3r_img_cond_finetune_complete/nova3r_training/version_37/test_scores.csv"

# Load the data
df = pd.read_csv(file_path)
display(df.head())

,CD_16384_full,f_score_0.1_full,f_score_0.05_full,f_score_0.02_full,seq_name,id
0,"(0.06945451349020004, 1)","(0.7480806112289429, 1)","(0.49004194140434265, 1)","(0.2498641163110733, 1)",replica_pano_large_apartment_2_003,0
1,"(0.030229508876800537, 1)","(0.9623070955276489, 1)","(0.8507229089736938, 1)","(0.4760282635688782, 1)",replica_pano_large_apartment_2_003,1
2,"(0.03329765796661377, 1)","(0.945345938205719, 1)","(0.8193447589874268, 1)","(0.45692014694213867, 1)",replica_pano_large_apartment_2_003,2
3,"(0.028302038088440895, 1)","(0.9620790481567383, 1)","(0.8626990914344788, 1)","(0.5283506512641907, 1)",replica_pano_large_apartment_2_003,3
4,"(0.03479893133044243, 1)","(0.943087100982666, 1)","(0.7931694984436035, 1)","(0.4799848198890686, 1)",replica_pano_large_apartment_2_003,4


In [3]:
# Identify the metric columns (everything except seq_name and id)
metric_cols = [col for col in df.columns if col not in ['seq_name', 'id']]

def parse_metric(val_str):
    """Parses a string like '(0.07, 1)' into a Python tuple."""
    if pd.isna(val_str):
        return 0.0, 0
    if isinstance(val_str, str):
        try:
            return ast.literal_eval(val_str)
        except Exception:
            return 0.0, 0
    return 0.0, 0

def get_weighted_average(series):
    """Calculates the weighted average for a pandas Series containing tuple strings."""
    parsed = series.apply(parse_metric)
    scores = parsed.apply(lambda x: x[0])
    weights = parsed.apply(lambda x: x[1])
    
    total_weight = weights.sum()
    if total_weight > 0:
        return (scores * weights).sum() / total_weight
    return 0.0

## Total Metrics
Calculations for the total combined weighted averages across all sequences.

In [4]:
print("--- Total Weighted Averages ---")
total_metrics = {}
for col in metric_cols:
    total_metrics[col] = get_weighted_average(df[col])
    print(f"{col}: {total_metrics[col]:.6f}")

--- Total Weighted Averages ---
CD_16384_full: 0.063706
f_score_0.1_full: 0.842366
f_score_0.05_full: 0.663112
f_score_0.02_full: 0.291711


## Per-Scene Metrics
Calculations for average metrics individually grouped by `seq_name`.

In [5]:
scene_results = []

for scene_name, group in df.groupby('seq_name'):
    row = {'seq_name': scene_name}
    # Count the number of total samples/items per scene
    row['num_samples'] = len(group)  
    for col in metric_cols:
        row[col] = get_weighted_average(group[col])
    scene_results.append(row)

scene_df = pd.DataFrame(scene_results)
display(scene_df)

,seq_name,num_samples,CD_16384_full,f_score_0.1_full,f_score_0.05_full,f_score_0.02_full
0,replica_pano_hotel_0_000,100,0.117110,0.631602,0.378493,0.130402
1,replica_pano_large_apartment_0_004,100,0.066638,0.853489,0.752555,0.310681
2,replica_pano_large_apartment_2_001,100,0.041819,0.943672,0.723446,0.254733
3,replica_pano_large_apartment_2_002,100,0.050033,0.878165,0.709610,0.361650
4,replica_pano_large_apartment_2_003,100,0.042931,0.904903,0.751456,0.401087
